# HR Applicant Shortlister

Upload your applicant Excel sheet, set **mandatory rules** (knock-outs) and **preferred cut-offs** (scoring), and watch the ranked table update live.

**How it works**
1. Run the cells from top to bottom.
2. In the last cell, upload your Excel file (or type its path) and click **Load**.
3. Change any rule. The table redraws after every change.
4. When you are happy, click **Finalize** and then **Download**.

**Rules**
- **Mandatory rule failed** → the applicant's score becomes **0**, they are placed **last**, and the *Reason* column says why (e.g. `Country 'USA' not allowed`).
- **Everyone else** gets a score out of 100 from the preferred targets and weights you set.

**Expected Excel columns** (names can vary slightly): `Name`, `Percentage`, `Experience`, `Skills`, `Permanent Residence` (or `Country`), `No. of Projects`.
`Skills` is one cell per applicant, like `Java 9/10, MySQL 9/10, Python 7/10` (the format `Java:9, MySQL:9` also works).

## 1. Setup

In [11]:
# pip install pandas openpyxl ipywidgets jinja2   (only if something is missing)
import re, io, os, sys, base64, html as _html
import pandas as pd
import ipywidgets as W
from IPython.display import display, clear_output

## 2. Helper functions (reading the sheet and scoring)

In [12]:
COLMAP_KEYS = {
    "Name":       ("name",),
    "Percentage": ("percent",),
    "Experience": ("exp",),
    "Skills":     ("skill",),
    "Country":    ("country", "resid", "nation"),
    "Projects":   ("project",),
}
SKILL_ITEM = re.compile(r"^(.+?)[\s:=\-]*(\d+(?:\.\d+)?)\s*(?:/\s*10)?\s*$")
SKILL_LABELS = {}          # "java" -> "Java" (keeps your original spelling)


def parse_skills(text):
    """'Java 9/10, MySQL:8' -> {'java': 9.0, 'mysql': 8.0}.  A skill with no rating counts as 0."""
    out = {}
    if pd.isna(text):
        return out
    for item in re.split(r"[,;\n]+", str(text)):
        item = item.strip()
        if not item:
            continue
        m = SKILL_ITEM.match(item)
        name, rating = (m.group(1).strip(), min(float(m.group(2)), 10.0)) if m else (item, 0.0)
        key = name.lower()
        SKILL_LABELS.setdefault(key, name)
        out[key] = rating
    return out


def clean_data(raw):
    cols, used = {}, set()
    for std, keys in COLMAP_KEYS.items():
        for c in raw.columns:
            if c not in used and any(k in str(c).lower() for k in keys):
                cols[std] = c
                used.add(c)
                break
    missing = [s for s in COLMAP_KEYS if s not in cols]
    if missing:
        raise ValueError(f"Could not find column(s) for {missing}. Columns in your file: {list(raw.columns)}")

    df = pd.DataFrame({std: raw[c] for std, c in cols.items()})
    df = df[df["Name"].notna() & (df["Name"].astype(str).str.strip() != "")].reset_index(drop=True)
    df["Name"] = df["Name"].astype(str).str.strip()
    df["Country"] = df["Country"].fillna("").astype(str).str.strip()
    for c in ("Percentage", "Experience", "Projects"):
        df[c] = pd.to_numeric(df[c].astype(str).str.replace("%", "", regex=False).str.strip(),
                              errors="coerce").fillna(0.0)
    if df["Percentage"].max() <= 1:          # 0.92 -> 92
        df["Percentage"] = df["Percentage"] * 100

    SKILL_LABELS.clear()
    sk = pd.DataFrame(list(df["Skills"].apply(parse_skills))).fillna(0.0)
    sk.index = df.index
    if sk.shape[1]:
        sk = sk[sk.gt(0).sum().sort_values(ascending=False).index]
    return df, sk


def fmt(x):
    return f"{x:g}"

In [13]:
state = {"df": None, "sk": None, "result": None, "total_w": 0.0, "finalized": False}
rows = {}          # fixed factors: Percentage, Experience, Projects
skill_rows = []    # skills HR adds while the app is running

country_sel = W.SelectMultiple(options=[], value=(), rows=6, layout=W.Layout(width="220px"))
country_add = W.Text(placeholder="type a country", continuous_update=False, layout=W.Layout(width="170px"))
country_add_btn = W.Button(description="Add country", icon="plus")
country_clear_btn = W.Button(description="Allow any country")
country_ui = W.HBox([
    W.VBox([W.HTML("<b>Allowed countries</b><br><small>Ctrl/Cmd-click to pick several.<br>"
                   "Nothing selected = any country.</small>"), country_sel]),
    W.VBox([W.HTML("<b>Add another country</b>"), country_add, country_add_btn, country_clear_btn],
           layout=W.Layout(margin="0 0 0 20px")),
])
strict_cb = W.Checkbox(value=False, indent=False,
                       description="Must-have means strictly greater than (>) instead of at least (≥)",
                       layout=W.Layout(width="650px"))
role_box = W.Text(value="Role", description="Role name", style={"description_width": "130px"},
                  layout=W.Layout(width="480px"))


def compute():
    df, sk = state["df"], state["sk"]
    n = len(df)
    strict = strict_cb.value
    allowed = {c.strip().lower() for c in country_sel.value}

    data = {"Percentage": df["Percentage"], "Experience": df["Experience"], "Projects": df["Projects"]}

    reasons = [[] for _ in range(n)]
    if allowed:
        for i in range(n):
            if df.at[i, "Country"].lower() not in allowed:
                reasons[i].append(f"Country '{df.at[i, 'Country'] or 'blank'}' not allowed")

    sign = "≤" if strict else "<"
    total_w = 0.0
    weighted = pd.Series(0.0, index=df.index)

    for key, r in rows.items():
        s = data[key]
        must, target, w = r["must"].value, r["target"].value, r["weight"].value
        if must > 0:
            fail = (s <= must) if strict else (s < must)
            for i in s.index[fail]:
                reasons[i].append(f"{r['label']} {fmt(s[i])} {sign} {fmt(must)}")
        if w > 0:
            denom = target if target > 0 else max(float(s.max()), 1.0)
            weighted += w * (s / denom).clip(0, 1)
            total_w += w

    not_found = []
    for sr in skill_rows:
        name = sr["name"].value.strip()
        if not name:
            continue
        key = name.lower()
        if key in sk.columns:
            s = sk[key]
        else:
            s = pd.Series(0.0, index=df.index)
            close = [SKILL_LABELS[k] for k in sk.columns if key in k or k in key]
            not_found.append(name + (f" (did you mean {', '.join(close)}?)" if close else ""))
        label = SKILL_LABELS.get(key, name)
        must, w = sr["must"].value, sr["weight"].value
        if must > 0:
            fail = (s <= must) if strict else (s < must)
            for i in s.index[fail]:
                reasons[i].append(f"{label} {fmt(s[i])} {sign} {must}")
        if w > 0:
            weighted += w * (s / 10.0).clip(0, 1)
            total_w += w

    score = (weighted / total_w * 100) if total_w > 0 else pd.Series(0.0, index=df.index)
    passed = pd.Series([len(r) == 0 for r in reasons], index=df.index)

    res = pd.DataFrame({
        "Name": df["Name"], "Country": df["Country"], "Percentage": df["Percentage"],
        "Experience": df["Experience"], "Projects": df["Projects"], "Skills": df["Skills"],
        "Score": score.where(passed, 0.0).round(1),
        "Status": passed.map({True: "Qualified", False: "Knocked out"}),
        "Reason": ["; ".join(r) for r in reasons],
    })
    res["_q"] = passed.values
    res = (res.sort_values(["_q", "Score"], ascending=[False, False], kind="stable")
              .drop(columns="_q").reset_index(drop=True))
    res.insert(0, "Rank", range(1, len(res) + 1))
    return res, total_w, not_found

## 3. Excel export

In [14]:
from openpyxl.styles import Font, PatternFill, Alignment

GREEN = PatternFill("solid", fgColor="E6F4EA")
RED = PatternFill("solid", fgColor="FCE8E6")


def _format_sheet(ws, df, colour=False):
    for c in ws[1]:
        c.font = Font(bold=True)
        c.alignment = Alignment(vertical="center")
    for j, col in enumerate(df.columns, start=1):
        longest = max([len(str(col))] + [len(str(v)) for v in df[col].head(300)])
        ws.column_dimensions[ws.cell(row=1, column=j).column_letter].width = min(longest + 3, 60)
    ws.freeze_panes = "A2"
    if colour and "Status" in df.columns:
        sidx = list(df.columns).index("Status")
        for r in range(2, len(df) + 2):
            fill = GREEN if ws.cell(row=r, column=sidx + 1).value == "Qualified" else RED
            for c in ws[r]:
                c.fill = fill


def build_excel(res):
    short = res[res["Status"] == "Qualified"]
    general = pd.DataFrame({
        "Setting": ["Role", "Allowed countries", "Must-have comparison"],
        "Value": [role_box.value, ", ".join(country_sel.value) or "any",
                  "strictly greater than (>)" if strict_cb.value else "at least (>=)"],
    })
    factors = pd.DataFrame([{"Factor": r["label"], "Must-have minimum": r["must"].value,
                             "Preferred target": r["target"].value, "Weight": r["weight"].value}
                            for r in rows.values()] +
                           [{"Factor": sr["name"].value.strip() + " (0-10)", "Must-have minimum": sr["must"].value,
                             "Preferred target": 10, "Weight": sr["weight"].value}
                            for sr in skill_rows if sr["name"].value.strip()])
    buf = io.BytesIO()
    with pd.ExcelWriter(buf, engine="openpyxl") as xw:
        res.to_excel(xw, sheet_name="Ranked", index=False)
        short.to_excel(xw, sheet_name="Shortlist", index=False)
        general.to_excel(xw, sheet_name="Rules used", index=False)
        factors.to_excel(xw, sheet_name="Rules used", index=False, startrow=len(general) + 2)
        _format_sheet(xw.sheets["Ranked"], res, colour=True)
        _format_sheet(xw.sheets["Shortlist"], short)
        ws = xw.sheets["Rules used"]
        ws.column_dimensions["A"].width = 26
        for col in "BCD":
            ws.column_dimensions[col].width = 22
        for c in ws[1]:
            c.font = Font(bold=True)
        for c in ws[len(general) + 3]:
            c.font = Font(bold=True)
    return buf.getvalue()

## 4. The app
Run this cell, then use the controls that appear below it. Nothing else to run.

In [15]:
summary = W.HTML()
table_out = W.Output()
download_out = W.Output()
finalize_msg = W.HTML()
status = W.Output()
rules_box = W.VBox()


def style_table(res):
    def colour(row):
        bg = "#e6f4ea" if row["Status"] == "Qualified" else "#fce8e6"
        return [f"background-color:{bg}; color:#202124"] * len(row)
    st = res.style.apply(colour, axis=1).format({"Score": "{:.1f}", "Percentage": "{:g}", "Experience": "{:g}",
                                                 "Projects": "{:g}"})
    try:
        st = st.hide(axis="index")
    except Exception:
        pass
    return st


def update_shares(total_w):
    for r in list(rows.values()) + skill_rows:
        counted = ("name" not in r) or r["name"].value.strip()
        w = r["weight"].value
        r["share"].value = f"{w / total_w * 100:.0f}%" if (total_w > 0 and w > 0 and counted) else "-"


def refresh():
    if state["df"] is None:
        return
    res, total_w, not_found = compute()
    update_shares(total_w)
    state["result"], state["total_w"] = res, total_w
    q = int((res["Status"] == "Qualified").sum())
    note = "" if total_w > 0 else "  <span style='color:#b3261e'>Give at least one factor a weight above 0 to get scores.</span>"
    if not_found:
        note += ("  <span style='color:#b3261e'>Not found in any applicant (counted as 0): "
                 + _html.escape(", ".join(not_found)) + "</span>")
    summary.value = (f"<b>{q}</b> qualified &nbsp;|&nbsp; <b>{len(res) - q}</b> knocked out (score 0, placed last) "
                     f"&nbsp;|&nbsp; {len(res)} applicants{note}")
    with table_out:
        clear_output(wait=True)
        display(style_table(res.head(300)))
        if len(res) > 300:
            print(f"Showing the first 300 of {len(res)} rows. The download contains everyone.")
    if state["finalized"]:
        state["finalized"] = False
        download_out.clear_output()
        finalize_msg.value = "<i>Rules changed, so the earlier download was cleared. Click Finalize again.</i>"


def on_change(_):
    refresh()


skills_box = W.VBox()
add_skill_btn = W.Button(description="Add skill", icon="plus", button_style="info")
add_all_btn = W.Button(description="Add all skills found in the file", icon="list")
hint = W.HTML()


def add_skill_row(name="", must=0, weight=5):
    name_w = W.Combobox(value=name, options=[SKILL_LABELS[k] for k in state["sk"].columns],
                        placeholder="type or pick a skill", ensure_option=False, continuous_update=False,
                        layout=W.Layout(width="200px"))
    must_w = W.IntSlider(value=must, min=0, max=10, continuous_update=False, readout=True,
                         layout=W.Layout(width="230px"))
    weight_w = W.IntSlider(value=weight, min=0, max=10, continuous_update=False, readout=True,
                           layout=W.Layout(width="230px"))
    del_w = W.Button(icon="trash", tooltip="Remove this skill", layout=W.Layout(width="40px"))
    share_w = W.HTML("-", layout=W.Layout(width="70px", margin="0 0 0 10px"))
    row = {"name": name_w, "must": must_w, "weight": weight_w, "share": share_w}
    box = W.HBox([name_w, must_w, weight_w, share_w, del_w])
    row["box"] = box

    def remove(_):
        skill_rows.remove(row)
        skills_box.children = [r["box"] for r in skill_rows]
        refresh()

    del_w.on_click(remove)
    for w_ in (name_w, must_w, weight_w):
        w_.observe(on_change, names="value")
    skill_rows.append(row)
    skills_box.children = [r["box"] for r in skill_rows]
    return row


def on_add_skill(_):
    add_skill_row()
    refresh()


def on_add_all(_):
    have = {r["name"].value.strip().lower() for r in skill_rows}
    for k in state["sk"].columns:
        if k not in have:
            add_skill_row(SKILL_LABELS[k])
    refresh()


add_skill_btn.on_click(on_add_skill)
add_all_btn.on_click(on_add_all)


def add_country(name):
    name = name.strip()
    if not name:
        return
    current = list(country_sel.value)
    opts = list(country_sel.options)
    match = [o for o in opts if o.lower() == name.lower()]
    if not match:
        opts.append(name)
        country_sel.options = opts      # changing options clears the selection, so restore it below
        match = [name]
    country_sel.value = tuple(dict.fromkeys(current + match))


def on_add_country(_):
    add_country(country_add.value)
    country_add.value = ""


def on_clear_country(_):
    country_sel.value = ()


country_add_btn.on_click(on_add_country)
country_clear_btn.on_click(on_clear_country)


def build_rules():
    rows.clear()
    found_countries = sorted({c for c in state["df"]["Country"] if c}, key=str.lower)
    country_sel.options = found_countries
    country_sel.value = tuple(c for c in found_countries if c.lower() == "india")
    skill_rows.clear()
    skills_box.children = []
    specs = [("Percentage", "Percentage", 0, 90, 2),
             ("Experience", "Experience (years)", 0, 3, 2),
             ("Projects", "Projects (count)", 0, 5, 1)]

    header = ["Factor", "Must-have minimum (0 = off)", "Preferred target", "Weight (0 = ignore)", "Share of score"]
    children = [W.HTML(f"<b>{h}</b>") for h in header]
    for key, label, must, target, weight in specs:
        boxes = [W.FloatText(value=v, layout=W.Layout(width="110px")) for v in (must, target, weight)]
        for b in boxes:
            b.observe(on_change, names="value")
        share = W.HTML("-")
        rows[key] = {"label": label, "must": boxes[0], "target": boxes[1], "weight": boxes[2], "share": share}
        children += [W.HTML(_html.escape(label)), *boxes, share]
    grid = W.GridBox(children, layout=W.Layout(grid_template_columns="210px 190px 130px 140px 110px",
                                                grid_gap="4px 12px"))
    for w_ in (country_sel, strict_cb):
        try:
            w_.unobserve(on_change, names="value")
        except Exception:
            pass
        w_.observe(on_change, names="value")

    found = ", ".join(SKILL_LABELS[k] for k in state["sk"].columns) or "none"
    hint.value = f"<i>Skills found in your file: {_html.escape(found)}</i>"
    skill_header = W.HBox([W.HTML("<b>Skill</b>", layout=W.Layout(width="200px")),
                           W.HTML("<b>Mandatory minimum /10 (0 = not mandatory)</b>", layout=W.Layout(width="230px")),
                           W.HTML("<b>Weightage /10 (0 = ignore)</b>", layout=W.Layout(width="230px")),
                           W.HTML("<b>Share</b>")])
    rules_box.children = [W.HTML("<h3>General rules</h3>"), role_box, country_ui, strict_cb,
                          W.HTML("<b>Weight</b> = how much a factor counts <i>compared with the others</i>. The Share column shows each factor's slice of the 100-point score: weights 25, 2 and 1 give shares of 89%, 7% and 4%. Use a similar scale for every factor (for example 1 to 10)."),
                          grid,
                          W.HTML("<h3>Skills</h3>"
                                 "<div>Add each skill you care about. A rating below its <b>mandatory minimum</b> "
                                 "gives the applicant a score of 0. The <b>weightage</b> decides how much the "
                                 "skill's rating counts in the score.</div>"),
                          hint, W.HBox([add_skill_btn, add_all_btn]), skill_header, skills_box]


# ---- load -----------------------------------------------------------
upload = W.FileUpload(accept=".xlsx,.xls", multiple=False, description="Upload Excel")
path_box = W.Text(value="hr_database.xlsx", description="or file path", layout=W.Layout(width="360px"))
load_btn = W.Button(description="Load", button_style="primary", icon="check")


def _uploaded_bytes():
    v = upload.value
    if not v:
        return None
    item = v[0] if isinstance(v, (tuple, list)) else list(v.values())[0]
    return bytes(item["content"])


def on_load(_):
    with status:
        clear_output()
        try:
            b = _uploaded_bytes()
            raw = pd.read_excel(io.BytesIO(b)) if b else pd.read_excel(path_box.value.strip())
            df, sk = clean_data(raw)
        except Exception as e:
            print("Could not load the file:", e)
            return
        print(f"Loaded {len(df)} applicants and {sk.shape[1]} distinct skills.")
    state.update(df=df, sk=sk, finalized=False)
    download_out.clear_output()
    finalize_msg.value = ""
    build_rules()
    refresh()


load_btn.on_click(on_load)

# ---- finalize / download --------------------------------------------
finalize_btn = W.Button(description="Finalize", button_style="success", icon="lock")


def on_finalize(_):
    if state["result"] is None:
        return
    data = build_excel(state["result"])
    safe = "".join(ch if ch.isalnum() else "_" for ch in role_box.value) or "role"
    fname = f"shortlist_{safe}.xlsx"
    with open(fname, "wb") as f:
        f.write(data)
    state["finalized"] = True
    finalize_msg.value = f"<b>Finalized.</b> Saved next to this notebook as <code>{_html.escape(fname)}</code>."
    mime = "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
    link = (f'<a download="{_html.escape(fname)}" href="data:{mime};base64,{base64.b64encode(data).decode()}" '
            f'style="display:inline-block;padding:8px 16px;background:#1a73e8;color:#fff;border-radius:6px;'
            f'text-decoration:none">Download {_html.escape(fname)}</a>')
    with download_out:
        clear_output()
        display(W.HTML(link))
        if "google.colab" in sys.modules:
            from google.colab import files
            files.download(fname)


finalize_btn.on_click(on_finalize)

app = W.VBox([
    W.HTML("<h3>Applicants</h3>"),
    W.HBox([upload, path_box, load_btn]),
    status,
    rules_box,
    W.HTML("<h3>Ranked table (live)</h3>"),
    summary,
    table_out,
    W.HTML("<h3>Finalize</h3>"),
    W.HBox([finalize_btn, finalize_msg]),
    download_out,
])
display(app)